In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer

portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

In [2]:
df = pd.read_csv('df_regex.csv', sep=';', dtype={'COMMENT_TEXT': str})

df = df.sample(10000)

In [33]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna()
    
    return df


def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

#######################################################################################################################
def classifica_label(df):
    
    # faz o drop dos na do dataframe de entrada
    df = df.dropna()
    
    # reseta o index do datraframe de entrada
    df2 = df.reset_index().drop('index',axis=1)
    
    # tokeniza os comentários do dataframe de entrada
    df2 = tokenizacao(df2)
    
    # faz uma cópia do dataframe de entrada
    df3 = df2.copy() 
    
    # reseta o index da cópia do dataframe
    df3 = df3.reset_index()
    
    # renomeia o index do dataframe cópia para usar o index_1 como a key do comentário
    df3 = df3.rename(columns={'index':'index_1'})

    # cria uma lista vazia com os comentários positivos
    lista_positivos = []

    # procura as frases que contém palavras positivas
    for n, objeto in enumerate(positivos_token):
        lista_1 = []
        for i in range(len(df2)):
            if (objeto[0] in df2.token[i]):
                lista_1.append([df2.token[i], df2.index[i]])
        for n, item in enumerate(lista_positivos):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_positivos.append(lista_1)

    # cria uma lista vazia com os comentários negativos
    lista_negativos = []

    # procura as frases que contém palavras negativas
    for n, objeto in enumerate(negativos_token):
        lista_2 = []
        for i in range(len(df2)):
            if (objeto[0] in df2.token[i]):
                lista_2.append([df2.token[i], df2.index[i]])
        for n, item in enumerate(lista_negativos):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_negativos.append(lista_2)
     
    # cria uma nova lista vazia com palavras positivas 
    lista_positivos_nova = []

    # Cria nova lista com palavras positivas usando o extend de listas
    for n, objeto in enumerate(lista_positivos):
        lista_temp = [item[0] for item in objeto] 
        lista_positivos_nova.extend(lista_temp)

    # cria uma nova lista vazia com palavras negativas
    lista_negativos_nova = []

    # Cria nova lista com palavras negativas usando o extend de listas
    for n, objeto in enumerate(lista_negativos):
        lista_temp = [item[0] for item in objeto] 
        lista_negativos_nova.extend(lista_temp)
    
    # cria uma lista vazia com palavras neutras
    lista_neutro = []
    
    # se a palavra não estiver nem na lista positiva nem na lista negativa, vira palavra neutra
    for i in range(len(df2)):
        if (df2.token[i] not in lista_positivos_nova) and (df2.token[i] not in lista_negativos_nova):
            lista_neutro.append([df2.token[i], df2.index[i]])
    
    # cria uma lista vazia de labels
    labels = []

    # gera o label positivo para comentários com palavras positivas
    for n, item in enumerate(lista_positivos):
        for i, objeto in enumerate(item):
            labels.append(2)

    # gera o label negativo para comentários com palavras negativas
    for n, item in enumerate(lista_negativos):
        for i, objeto in enumerate(item):
            labels.append(1)

    # gera o label neutro para comentários com palavras neutras
    for n, item in enumerate(lista_neutro):
            labels.append(0)
     
    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos positivos
    df_positivo = pd.DataFrame(lista_positivos[0])
    
    # concatena o resto da lista para sentimentos positivos
    for n, objeto in enumerate(lista_positivos[1:]):
        df_positivo = pd.concat([df_positivo,pd.DataFrame(objeto)])

    # renomeia as colunas do dataframe criado para sentimentos positivos
    df_positivo = df_positivo.rename(columns=({0:'token',1:'index_1'})) 

    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos negativos
    df_negativo = pd.DataFrame(lista_negativos[0])
    
    # concatena o resto da lista para sentimentos negativos
    for n, objeto in enumerate(lista_negativos[1:]):
        df_negativo = pd.concat([df_negativo,pd.DataFrame(objeto)])
    
    # renomeia as colunas do dataframe criado para sentimentos negativos
    df_negativo = df_negativo.rename(columns={0:'token',1:'index_1'})

    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos neutros
    df_neutro = pd.DataFrame(lista_neutro)
    
    # renomeia as colunas do dataframe criado para sentimentos neutros
    df_neutro = df_neutro.rename(columns={0:'token',1:'index_1'})

    # concatena os dataframes positivos e negativos
    df_pos_neg = pd.concat([df_positivo,df_negativo])
    
    # concatena com o dataframe neutro
    df_pos_neg_neutro = pd.concat([df_pos_neg,df_neutro])

    # cria a coluna com os labels definidos anteriormente
    df_pos_neg_neutro['labels'] = labels

    # Agrupa comentários classificados com ambiguidade (possivelmente ironias, e.g gostei muito... horrível)
    # df_pos_neg_neutro = df_pos_neg_neutro.groupby('index_1').agg(token=('token','first'),
    #                             labels=('labels','first')).reset_index()
    
    #
    columns_keep = df.columns.to_list()
    
    #
    columns_keep.append('index_1')
    
    # faz o merge entre os dataframes 
    df_final = pd.merge(df_pos_neg_neutro,
                        df3[columns_keep], 
                        how = 'left',
                        left_on = 'index_1',
                        right_on = 'index_1'
                       )
    
   
    columns_keep_final = columns_keep
    
    columns_keep_final.extend(['labels','token'])
    
    # renomeia o nome da coluna labels para sentimento_labels
    df_final = df_final[columns_keep_final].rename(columns={'labels': 'sentimento_labels'})
    
    # aplica o dicionário de sentimentos
    df_final.sentimento_labels = df_final.sentimento_labels.map(dic_sentimentos) 
    
    return df_final

In [34]:
palavras_positivas = ['bom',
                      'lindo',
                      'fantastico',
                      'top',
                      'maravilha',
                      'amou',
                      'amando',
                      'melhor',
                      'parabéns',
                      'elogio',
                      'incrível',
                      'adorei',
                      'excelente',
                      'certinho',
                      'perfeito',
                      'gostei muito',
                      'ótimo'
                     ]

palavras_negativas = ['fragil',
                      'insatisfeito',
                      'horrível',
                      'pessimo',
                      'lixo',
                      'bosta',
                      'devagar'
                     ]

dic_sentimentos = {
    2:'Positivo',
    1:'Negativo',
    0:'Neutro'
}

#
positivos_token = [preprocess_text(item) for item in palavras_positivas]

#
negativos_token = [preprocess_text(item) for item in palavras_negativas]

In [35]:
#pd.set_option("max_rows", None)
pd.options.display.max_colwidth = 200

df_final = classifica_label(df)

df_final

,AP2,COUNTRY,COMMENT_TEXT,index_1,sentimento_labels,token
0,SEDA,BR,"Uma boa TV. Comprei para usá-la como meu monitor direto do computador, muito bom!",4,Positivo,"[boa, comprei, usa, monitor, direto, computador, bom]"
1,SEDA,BR,"aparelho muito bom,as fotos de ótima qualidade.porem muito satisfeita",6,Positivo,"[aparelho, bom, fotos, otima, qualidade, porem, satisfeita]"
2,SEDA,BR,"Gostei, acabamento muito bom, resfria os produtos de acordo que eu queria, muito satisfeito.",14,Positivo,"[gostei, acabamento, bom, resfria, produtos, acordo, queria, satisfeito]"
3,SEDA,BR,Muito bom. Fácil de usar. Fiel a descrição do anúncio. Sei que é bbom pois já havia comprado outro no lançamento\r\nE fiquei satisfeita\r\n,17,Positivo,"[bom, facil, usar, fiel, descricao, anuncio, sei, bbom, pois, havia, comprado, outro, lancamento, fiquei, satisfeita]"
4,SEDA,BR,"Produto bom, dentro de suas limitações da p usar no dia-a-dia tranquilo, intermediário de boa qualidade.",19,Positivo,"[produto, bom, dentro, limitacoes, usar, dia, dia, tranquilo, intermediario, boa, qualidade]"
...,...,...,...,...,...,...
10022,SEDA,BR,"A Geladeira é linda, espaço suficiente armazenar todos meus mantimentos.\r\nGela super rápido!!!! Ela é uma verdadeira Brastemp!!!",9989,Neutro,"[geladeira, linda, espaco, suficiente, armazenar, todos, mantimentos, ngela, super, rapido, verdadeira, brastemp]"
10023,SEDA,BR,Acho que hoje o redmi note 8 está sendo uma das melhores opções a custo benefício.,9990,Neutro,"[acho, hoje, redmi, note, sendo, melhores, opcoes, custo, beneficio]"
10024,SEDA,BR,"Celular ótimo, atende a tudo aquilo que se propõe.",9992,Neutro,"[celular, otimo, atende, tudo, propoe]"
10025,SEDA,BR,Electrolux é sempre de confiança... Lavadora silenciosa e de simples manuseio. Perfeita,9994,Neutro,"[electrolux, sempre, confianca, lavadora, silenciosa, simples, manuseio, perfeita]"


In [37]:
df_novo = pd.read_csv('df_novo.csv',sep=';',encoding='utf-8')

df_novo = df_novo.sample(10)

C:\ProgramData\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3071: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [38]:
#pd.set_option("max_rows", None)
pd.options.display.max_colwidth = 200

df_ = classifica_label(df_novo)

df_

,KEY,COMMENT_ID,COMMENT_TEXT,index_1,sentimento_labels,token
0,aHR0cHM6Ly93d3cuZXh0cmEuY29tLmJyL3NtYXJ0cGhvbmUtbW90b3JvbGEtb25lLWZ1c2lvbi1henVsLWluZGlnby0xMjhnYi10ZWxhLWRlLTYtNS00Z2ItcmFtLWNhbWVyYS10cmFzZWlyYS1xdWFkcnVwbGEtYW5kcm9pZC0xMC1lLXByb2Nlc3NhZG9yLXF1...,8870131.0,Bom aparelho de celular,2,Positivo,"[bom, aparelho, celular]"
1,aHR0cHM6Ly93d3cubWVyY2Fkb2xpdnJlLmNvbS5ici9sZy1rNTItMTMtbXB4LWR1YWwtc2ltLTY0LWdiLWdyYXktMy1nYi1yYW0vcC9NTEIxNjk1MTEwNzEwMzM0NjEyOA==,103346128.0,"Bom produto, mas como o fabricante está saindo do mercado a compra deve ser vem pensada.",6,Positivo,"[bom, produto, fabricante, saindo, mercado, compra, deve, ser, vem, pensada]"
2,aHR0cHM6Ly93d3cuYW1lcmljYW5hcy5jb20uYnIvcHJvZHV0by8xMzQwNjE3ODEyMDE5LTExLTE0VDAxOjEwOjIyLjAwMCswMDowMENhbWVyYSBuYW8gbXVpdG8gYm9hICwsLCBjZWx1bGFyIGJlbSBzaW1wbGVzLi4uIFBhcmVjaWEgc2VyIG1lbGhvcg==,214385481,"Camera nao muito boa ,,, celular bem simples... Parecia ser melhor",1,Positivo,"[camera, nao, boa, celular, bem, simples, parecia, ser, melhor]"
3,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zbWFydHBob25lLXNhbXN1bmctZ2FsYXh5LWEwMS1wcmV0by0zMmdiLXRlbGEtaW5maW5pdGEtZGUtNS03LWNhbWVyYS10cmFzZWlyYS1kdXBsYS1hbmRyb2lkLTEwLTAtZHVhbC1jaGlwLWUtcHJvY2Vzc2Fk...,8048014.0,Entrega rápida produto ótimo adorei super recomendo.,9,Positivo,"[entrega, rapida, produto, otimo, adorei, super, recomendo]"
4,aHR0cHM6Ly93d3cuYW1lcmljYW5hcy5jb20uYnIvcHJvZHV0by8xNDQxMDc5NDk4MjAyMS0wOS0xNVQxMzowMjoxOS4wMDArMDA6MDBRdWFsaWRhZGUgZGUgaW1hZ2VtIGV4Y2VsZW50ZSwgcmVzcG9uZGUgcuFwaWRvLiBSZWNvbWVuZG8gc2VtIGT6dmlkYXMh...,234204811,"Qualidade de imagem excelente, responde rápido. Recomendo sem dúvidas!!",4,Positivo,"[qualidade, imagem, excelente, responde, rapido, recomendo, duvidas]"
5,aHR0cHM6Ly93d3cuYW1lcmljYW5hcy5jb20uYnIvcHJvZHV0by8yOTE3MjY5NDU4MjAyMS0wNS0yN1QyMzo0Njo1Mi4wMDArMDA6MDDTdGltbyBjZWx1bGFyLCBjYW1lcmEg83RpbWEsIG1hcyBuYW8gdGVtIG9zIDY0TVAgZGUgY2FtZXJhIHF1ZSBwcm9tZXRl...,231489875,"Ótimo celular, camera ótima, mas nao tem os 64MP de camera que prometem",0,Neutro,"[otimo, celular, camera, otima, nao, camera, prometem]"
6,aHR0cHM6Ly93d3cucG9udG9mcmlvLmNvbS5ici9zbWFydHBob25lLXNhbXN1bmctZ2FsYXh5LXMyMC1henVsLTEyOGdiLThnYi1yYW0tdGVsYS1pbmZpbml0YS1kZS02LTItY2FtZXJhLXRyaXBsYS10cmFzZWlyYS1jYW1lcmEtZnJvbnRhbC0xMG1wLWlwNjgt...,8876785.0,"Esse celular é maravilhoso! E para mim, o tamanho ideal!",3,Neutro,"[celular, maravilhoso, mim, tamanho, ideal]"
7,aHR0cHM6Ly93d3cubWVyY2Fkb2xpdnJlLmNvbS5ici9tb3RvLWc2MC1kdWFsLXNpbS0xMjgtZ2ItYXp1bC02LWdiLXJhbS9wL01MQjE3OTQ5NTUxMTIwNzM3Mzgw,120737380,Celular que atende todas as expectativas mas o carregamento demora demais acho que o carregador é muito fraco pra uma bateria tão potente.,5,Neutro,"[celular, atende, todas, expectativas, carregamento, demora, demais, acho, carregador, fraco, pra, bateria, tao, potente]"
8,aHR0cHM6Ly93d3cubWVyY2Fkb2xpdnJlLmNvbS5ici9zYW1zdW5nLWdhbGF4eS1hMTBzLWR1YWwtc2ltLTMyLWdiLXByZXRvLTItZ2ItcmFtL3AvTUxCMTUxNTcyOTQxMDkyNDMzNjg=,109243368.0,"Ainda não tive tempo de usar o produto, mais os testes,por mim realizados, o aparelho cumpri o que promete.",7,Neutro,"[ainda, nao, tempo, usar, produto, testes, mim, realizados, aparelho, cumpri, promete]"
9,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcnQtdHYtbGVkLTMyLWhkLWxnLTMybG02MjVicHNiLXRoaW5xLWFpLWludGVsaWdlbmNpYS1hcnRpZmljaWFsLWNvbS1pb3QtdmlydHVhbC1zdXJyb3VuZC1zb3VuZC13ZWJvcy00LTUtaGRyLXF1YWQt...,6322529.0,Boa só o volume para música que é baixo .,8,Neutro,"[boa, volume, musica, baixo]"
